<a href="https://colab.research.google.com/github/fur-ruf/data_analysis/blob/main/lab3_mashine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа №3
## Ассоциативные правила

Выполнила: Будюкова Полина (408326)

Группа: P3321

Преподаватель: Солодкая Мария Александровна


### Цель работы:
Изучить алгоритмы ассоциативных правил на примере набора данных с транзакциями людей в розничной торговле.

### Задание:
1. Реализовать алгоритмы Apriori и FP-Growth.
2. Оценить время обучения каждого из алгоритмов.
3. Реализовать функцию проверки путем вывода получившихся правил как рекомендаций к текущему содержимому потребительской корзины.

## Получение данных

In [ ]:
def load_data(path, top_n=100):
    for sep in [',', ';', '\t']:
        try:
            df = pd.read_csv(path, encoding='latin1', sep=sep, on_bad_lines='skip')
            if len(df.columns) > 2:
                break
        except:
            continue

    df = df[df['Quantity'] > 0]
    df = df.dropna(subset=['CustomerID', 'Description'])

    df['Description'] = df['Description'].str.strip().str.upper()
    df = df.drop_duplicates(['InvoiceNo', 'Description'])

    top_items = df['Description'].value_counts().head(top_n).index
    df = df[df['Description'].isin(top_items)]

    transactions = df.groupby('InvoiceNo')['Description'].apply(set).tolist()
    return transactions

## Реализация алгоритмов

In [ ]:
def support(transactions, itemset):
  count = 0
  for t in transactions:
    if itemset.issubset(t):
      count += 1
  return count / len(transactions)

def apriori(transactions, min_sup):
    items = set(i for t in transactions for i in t)

    current = [frozenset([i]) for i in items]
    freq = []

    k = 1
    while current:
        next_level = []

        for itemset in current:
            sup = support(transactions, itemset)
            if sup >= min_sup:
                freq.append((itemset, sup))
                next_level.append(itemset)

        new_candidates = set()
        for i in range(len(next_level)):
            for j in range(i + 1, len(next_level)):
                union = next_level[i] | next_level[j]
                if len(union) == k + 1:
                    new_candidates.add(union)

        current = list(new_candidates)
        k += 1

    return freq

In [ ]:
class FPNode:
    def __init__(self, item, count, parent):
        self.item = item
        self.count = count
        self.parent = parent
        self.children = {}
        self.link = None


def build_fp_tree(transactions, min_sup):
    from collections import defaultdict

    def unpack(t):
        return t if isinstance(t, tuple) else (t, 1)

    n = sum(unpack(t)[1] for t in transactions)

    item_count = defaultdict(int)
    for t in transactions:
        items, count = unpack(t)
        for i in items:
            item_count[i] += count

    item_count = {i: c for i, c in item_count.items() if c / n >= min_sup}
    if not item_count:
        return None, None

    def sort_items(t):
        items, count = unpack(t)
        items = [i for i in items if i in item_count]
        items.sort(key=lambda x: (-item_count[x], x))
        return items, count

    root = FPNode(None, 0, None)
    header = {i: None for i in item_count}

    for t in transactions:
        items, count = sort_items(t)
        current = root

        for item in items:
            if item in current.children:
                current.children[item].count += count
            else:
                node = FPNode(item, count, current)
                current.children[item] = node

                if header[item] is None:
                    header[item] = node
                else:
                    cur = header[item]
                    while cur.link:
                        cur = cur.link
                    cur.link = node

            current = current.children[item]

    return root, header


def mine_tree(header, min_sup, prefix, freq_itemsets, n, max_len=None):
    items = sorted(header.keys(), key=lambda x: x)

    for item in items:
        new_prefix = prefix | {item}

        if max_len and len(new_prefix) > max_len:
            break

        support = 0
        node = header[item]
        while node:
            support += node.count
            node = node.link

        sup_val = support / n
        freq_itemsets.append((frozenset(new_prefix), sup_val))

        cond_patterns = []
        node = header[item]

        while node:
            path = []
            parent = node.parent

            while parent and parent.item is not None:
                path.append(parent.item)
                parent = parent.parent

            if path:
                cond_patterns.append((path, node.count))

            node = node.link

        cond_tree, cond_header = build_fp_tree(cond_patterns, min_sup)

        if cond_header:
            mine_tree(cond_header, min_sup, new_prefix, freq_itemsets, n, max_len)


def fpgrowth(transactions, min_sup, max_len=3):
    root, header = build_fp_tree(transactions, min_sup)

    if not header:
        return []

    freq_itemsets = []
    n = len(transactions)

    mine_tree(header, min_sup, set(), freq_itemsets, n, max_len)

    return freq_itemsets

## Генерация правил, составление рекомендаций и вывод статистики

In [ ]:
def generate_rules_with_metrics(freq, min_conf, transactions):
    sup_dict = dict(freq)
    n = len(transactions)
    rules = []

    for itemset, sup in freq:
        if len(itemset) < 2:
            continue

        for item in itemset:
            A = itemset - {item}
            B = frozenset([item])

            if A in sup_dict:
                conf = sup / sup_dict[A]
                if conf >= min_conf:
                    sup_B = sup_dict.get(B, 0)
                    lift = conf / sup_B if sup_B > 0 else 0
                    leverage = sup - sup_dict[A] * sup_B

                    rules.append((A, B, conf, lift, leverage))

    return rules

In [ ]:
def recommend(cart, rules):
    cart = set(cart)
    rec_dict = {}

    for rule in rules:
        if len(rule) >= 5:
            A, B, conf, lift, leverage = rule[:5]
        else:
            continue

        if len(A & cart) > 0 and not B.issubset(cart):
            item = list(B)[0]

            if item not in rec_dict or rec_dict[item] < conf:
                rec_dict[item] = conf

    recs = sorted(rec_dict.items(), key=lambda x: x[1], reverse=True)

    if recs:
        print("\nРекомендации:")
        for i, (item, conf) in enumerate(recs[:5], 1):
            print(f"  {i}. {item} (уверенность: {conf:.4f})")
    else:
        print("\nНет рекомендаций для данной корзины")

    return recs[:5]

In [ ]:
def print_rules_statistics(rules, top_n=10):
    if not rules:
        print("Нет правил")
        return

    if len(rules[0]) >= 5:
        print(f"\n{'='*80}")
        print(f"Статистика по правилам:")
        print(f"{'='*80}")

        confidences = [r[2] for r in rules]
        lifts = [r[3] for r in rules]
        leverages = [r[4] for r in rules]

        print(f"\nВсего правил: {len(rules)}")
        print(f"Средняя уверенность: {sum(confidences)/len(confidences):.4f}")
        print(f"Максимальная уверенность: {max(confidences):.4f}")
        print(f"Минимальная уверенность: {min(confidences):.4f}")
        print(f"Средний Lift: {sum(lifts)/len(lifts):.4f}")
        print(f"Максимальный Lift: {max(lifts):.4f}")
        print(f"Правил с Lift > 1: {sum(1 for l in lifts if l > 1)}")
        print(f"Правил с Lift <= 1: {sum(1 for l in lifts if l <= 1)}")

        print(f"\n{'='*80}")
        print(f"Топ-{top_n} правил по Lift (наиболее интересные):")
        print(f"{'='*80}")

        sorted_rules = sorted(rules, key=lambda x: x[3], reverse=True)

        for i, rule in enumerate(sorted_rules[:top_n], 1):
            A, B, conf, lift, leverage = rule
            A_str = ', '.join(list(A)[:3])
            if len(A) > 3:
                A_str += f" и еще {len(A)-3}"
            B_str = list(B)[0]

            print(f"\n{i}. {A_str} → {B_str}")
            print(f"   Уверенность: {conf:.4f}")
            print(f"   Lift: {lift:.4f} {'(положительная корреляция)' if lift > 1 else '(отрицательная корреляция)'}")
            print(f"   Leverage: {leverage:.4f}")
    else:
        print(f"\nВсего правил: {len(rules)}")
        print("\nПервые 5 правил:")
        for i, rule in enumerate(rules[:5], 1):
            A, B, conf = rule
            A_str = ', '.join(list(A)[:3])
            if len(A) > 3:
                A_str += f" и еще {len(A)-3}"
            B_str = list(B)[0]
            print(f"{i}. {A_str} → {B_str} (conf={conf:.4f})")

## Основная часть

In [ ]:
import pandas as pd
from collections import defaultdict
from itertools import combinations
import time
from collections import defaultdict

transactions = load_data("sample_data/OnlineRetail.csv")

min_sup = 0.005
min_conf = 0.3

print("\nЗапуск Apriori:")
t1 = time.time()
freq1 = apriori(transactions, min_sup)
time_apriori = time.time() - t1
print(f"Apriori: {time_apriori:.2f} секунд")
print(f"Найдено частых наборов: {len(freq1)}")

print("\nЗапуск FP-Growth:")
t2 = time.time()
freq2 = fpgrowth(transactions, min_sup)
time_fpgrowth = time.time() - t2
print(f"FP-Growth: {time_fpgrowth:.2f} секунд")
print(f"Найдено частых наборов: {len(freq2)}")

print("СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ:")
print(f"FP-Growth быстрее Apriori в {time_apriori/time_fpgrowth:.1f} раз")

rules = generate_rules_with_metrics(freq2, min_conf, transactions)

print_rules_statistics(rules, top_n=10)

cart = ["REGENCY CAKESTAND 3 TIER"]
print(f"\nКорзина: {cart}")

recs = recommend(cart, rules)

print("Правила, использованные для рекомендаций:")

cart_set = set(cart)
relevant_rules = []
for rule in rules:
    A, B, conf, lift, leverage = rule[:5]
    if len(A & cart_set) > 0 and not B.issubset(cart_set):
        relevant_rules.append(rule)

if relevant_rules:
    relevant_rules.sort(key=lambda x: x[2], reverse=True)
    for i, rule in enumerate(relevant_rules[:5], 1):
        A, B, conf, lift, leverage = rule[:5]
        A_str = ', '.join(list(A))
        B_str = list(B)[0]
        print(f"{i}. {A_str} → {B_str}")
        print(f"   Уверенность: {conf:.4f}, Lift: {lift:.4f}")
else:
    print("Нет подходящих правил")


Запуск Apriori:
Apriori: 45.91 секунд
Найдено частых наборов: 2501

Запуск FP-Growth:
FP-Growth: 20.27 секунд
Найдено частых наборов: 160083
СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ:
FP-Growth быстрее Apriori в 2.3 раз

Статистика по правилам:

Всего правил: 41879
Средняя уверенность: 0.4142
Максимальная уверенность: 1.0000
Минимальная уверенность: 0.3000
Средний Lift: 7.4076
Максимальный Lift: 25.3628
Правил с Lift > 1: 41879
Правил с Lift <= 1: 0

Топ-10 правил по Lift (наиболее интересные):

1. SMALL WHITE HEART OF WICKER, GREEN REGENCY TEACUP AND SAUCER → PINK REGENCY TEACUP AND SAUCER
   Уверенность: 0.9333
   Lift: 25.3628 (положительная корреляция)
   Leverage: 0.0009

2. GREEN REGENCY TEACUP AND SAUCER, JUMBO BAG ALPHABET → PINK REGENCY TEACUP AND SAUCER
   Уверенность: 0.9167
   Lift: 24.9099 (положительная корреляция)
   Leverage: 0.0014

3. GREEN REGENCY TEACUP AND SAUCER, JUMBO  BAG BAROQUE BLACK WHITE → PINK REGENCY TEACUP AND SAUCER
   Уверенность: 0.8889
   Lift: 24.1551 (положител